# 04 仮想配分のリスクを分解する

## Goal

共通APIの virtual_risk_report を使い、2銘柄の仮想ウェイト、残現金、年率ボラティリティと寄与を確認します。実口座の保有・取引は読みません。

## Setup

- 入力: 固定合成デモの60営業日、DEMO:ALPHAとDEMO:BETA、USD / raw
- 使用モード: retrospective。現在の構成を過去窓に当てた感応度でありPIT戦略評価ではありません
- ウェイト: 40% / 35%、現金25%。各銘柄上限50%、最低現金20%
- 直近20リターン、年率換算252期。日次標本共分散を利用
- 欠損を前値で補完せず、通貨換算をしません。デモの判断時刻はUTCです

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from market_research.research.indicators import IndicatorInput
from market_research.research.portfolio import VirtualConstraints, virtual_risk_report
from market_research.services import build_demo_run

run = build_demo_run()
source = IndicatorInput(run.prices, "retrospective", "USD", "raw")
weights = pd.Series({"DEMO:ALPHA": 0.40, "DEMO:BETA": 0.35})
print(f"run_id={run.run_id}, as_of={run.prices.index[-1].isoformat()}, assets={len(weights)}")

## Steps

### 1. 制約を満たす仮想配分のリスクを計算する

市場側の研究近似です。口座残高、税額、現実の為替取引を表しません。

In [ ]:
risk = virtual_risk_report(
    source,
    weights,
    base_currency="USD",
    lookback=20,
    periods_per_year=252,
    constraints=VirtualConstraints(
        max_gross=1.0, max_name_weight=0.50, cash_min=0.20,
    ),
)
contributions = pd.DataFrame(
    {
        "weight": risk.weights,
        "annualized_vol_contribution_pct_points": 100 * risk.component_vol_annualized,
    }
)
print(
    f"window={risk.start_at.isoformat()} to {risk.end_at.isoformat()}, "
    f"cash={risk.cash_weight:.1%}, annualized_volatility={risk.total_vol_annualized:.3%}"
)
contributions.round(4)

### 2. ボラティリティ寄与を確認する

縦軸は年率ボラティリティへの寄与のパーセントポイントです。銘柄別寄与は負になることもあります。ここでは合成データの符号をそのまま示します。

In [ ]:
figure = go.Figure(
    go.Bar(
        x=risk.component_vol_annualized.index,
        y=100 * risk.component_vol_annualized.values,
        text=[f"{100 * value:.3f}" for value in risk.component_vol_annualized.values],
        textposition="outside",
    )
)
figure.update_layout(
    title="仮想配分の年率ボラティリティ寄与 (直近20リターン)",
    xaxis_title="合成銘柄",
    yaxis_title="年率ボラティリティ寄与 (パーセントポイント)",
    template="plotly_white",
)
figure

## Checks

寄与の合計はポートフォリオの年率ボラティリティに一致します。品質と欠損の検査は virtual_risk_report が行い、欠損があれば計算を止めます。

In [ ]:
assert risk.mode == "retrospective"
assert risk.base_currency == "USD" and risk.adjustment == "raw"
assert risk.cash_weight == 0.25
assert abs(risk.component_vol_annualized.sum() - risk.total_vol_annualized) < 1e-12
assert risk.lookback == 20 and risk.periods_per_year == 252
print(f"年率ボラティリティ={risk.total_vol_annualized:.3%}、現金={risk.cash_weight:.1%}")

## Next Steps

05では同じ合成runを再計算せずにHTMLへレンダリングします。04のretrospectiveリスク表はそのHTMLに含まれません。